# Postprocessing core

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/postprocessing/Document_Postprocessing_Core.ipynb)


## Jump to an API

- [postprocessing.Operation](#postprocessingoperation)
- [postprocessing.Pipeline](#postprocessingpipeline)
- [postprocessing.compose](#postprocessingcompose)
- [postprocessing.apply](#postprocessingapply)
- [postprocessing.custom](#postprocessingcustom)

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@8e42b64

In [ ]:
import tempfile
from PIL import Image
from klygo import postprocessing as post, files
from klygo.outputs.detect import Box, Detection, Detections
tutorial_workspace = tempfile.TemporaryDirectory()
root = files.path(tutorial_workspace.name)
image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])


### postprocessing.Operation

`postprocessing.Operation(name: 'str', function: 'Callable[[Any], Any]') -> None`

Name and apply one custom result transformation.

- Parameters: name labels the operation; function transforms a result.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
operation = post.Operation("keep_cats", lambda result: post.select(result, labels=["cat"]))
print(operation(frame).labels)

**Practical variation**

In [ ]:
print(post.Operation("identity", lambda value: value)(frame) is frame)

**Expected error and recovery**

In [ ]:
try:
    post.Operation('invalid', None)(frame)
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.Pipeline

`postprocessing.Pipeline(operations: 'Iterable[Callable[[Any], Any]]') -> 'None'`

Apply an ordered series of result transformations.

- Parameters: operations is an iterable of callables or nested pipelines.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
pipeline = post.Pipeline([post.threshold(0.5), post.top(1)])
print(pipeline(frame).labels)

**Practical variation**

In [ ]:
print(len(post.Pipeline([post.clip(), post.threshold(0.5)])))

**Expected error and recovery**

In [ ]:
try:
    post.Pipeline([42])
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.compose

`postprocessing.compose(*operations: 'Callable[[Any], Any]') -> 'Pipeline'`

Build a reusable Pipeline from operations.

- Parameters: operations are callables passed individually or as one iterable.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
pipeline = post.compose(post.threshold(0.5), post.top(1))
print(pipeline(frame).labels)

**Practical variation**

In [ ]:
print(len(post.compose([post.clip(), post.top(2)])))

**Expected error and recovery**

In [ ]:
try:
    post.compose(42)
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.apply

`postprocessing.apply(result: 'Any', *operations: 'Callable[[Any], Any]') -> 'Any'`

Transform a result immediately with one or more operations.

- Parameters: result is a Detection or Detections; operations run in order.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
print(post.apply(frame, post.threshold(0.5), post.top(1)).labels)

**Practical variation**

In [ ]:
print(post.apply(frame).labels)

**Expected error and recovery**

In [ ]:
try:
    post.apply(object(), post.top(1))
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

### postprocessing.custom

`postprocessing.custom(function: 'Callable[[Any], Any]', name: 'Optional[str]' = None) -> 'Operation'`

Wrap a callable as a named operation.

- Parameters: function is callable; name optionally overrides its display name.
- Returns: A new result, operation, pipeline, or IoU score according to the signature.

**Basic use**

In [ ]:
from PIL import Image
from klygo import postprocessing as post
from klygo.outputs.detect import Box, Detection

image = Image.new("RGB", (100, 80), "white")
frame = Detection(image, [Box(0, "cat", 0.9, [-5, 0, 50, 50]), Box(0, "cat", 0.8, [0, 0, 48, 48]), Box(1, "dog", 0.4, [60, 10, 90, 60])])
operation = post.custom(lambda result: post.select(result, labels=["cat"]), name="cats")
print(operation(frame).labels)

**Practical variation**

In [ ]:
print(post.custom(lambda value: value).name)

**Expected error and recovery**

In [ ]:
try:
    post.custom(42)
except TypeError as error:
    print(type(error).__name__, 'Correct the input and retry')

## Put it together

In [ ]:
pipeline = post.compose(post.clip(), post.threshold(0.5), post.top(1))
print(pipeline(frame).labels)

## Remember

Postprocessing returns new results and keeps frame streams lazy. Continue with [Detect](Document_Postprocessing_Detect.ipynb) or the [overview](Document_Postprocessing_Overview.ipynb). Related guides: [Outputs](../outputs/Document_Outputs_Overview.ipynb) and [Visual](../visual/Document_Visual_Overview.ipynb).

In [ ]:
tutorial_workspace.cleanup()